In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv") #-data.csv")

In [2]:
df_ground_truth.head()

,question,document
0,I just found this course late — can I still jo...,74eb249bbf
1,If I join after the course has already started...,74eb249bbf
2,What do I need to do to qualify for a certific...,74eb249bbf
3,Is it okay to start the course after it begins...,74eb249bbf
4,"When I join late, do I only need to submit the...",74eb249bbf


In [3]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': 'db78580409',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often used in text analysis to determine how similar two documents are based on their content. This metric computes the cosine of the angle between two vectors, which are typically word counts or TF-IDF values of the documents. The cosine similarity value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates that the vectors are orthogonal (no similarity), and -1 represents completely opposite vectors.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](h

In [7]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [15]:
q = ground_truth[1]
q

{'question': 'If I join after the course has already started, am I still able to get a certificate?',
 'document': '74eb249bbf'}

In [18]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [20]:
results = text_search(q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\n

In [21]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
9f689c185f == 74eb249bbf: False
69d122f12e == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
cdc3b285e5 == 74eb249bbf: False


In [22]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [23]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [24]:
q = ground_truth[1]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

If I join after the course has already started, am I still able to get a certificate?


[1, 0, 0, 0, 0]

In [26]:
q = ground_truth[16]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where do I find the docs, repo, and deadlines for LLM Zoomcamp?


[0, 0, 0, 0, 0]

In [27]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [28]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/515 [00:00<?, ?it/s]

In [31]:
relevance[:15]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [32]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [33]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [34]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/515 [00:00<?, ?it/s]

In [35]:
sample = relevance_total[:15]

In [36]:
sample

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [37]:
14 / 15

0.9333333333333333

In [38]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.9333333333333333

In [39]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [40]:
hit_rate(relevance)

0.8466019417475729

In [41]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.7633333333333333

In [42]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [43]:
mrr(sample)

0.7633333333333333

In [44]:
mrr(relevance)

0.7127184466019416

In [45]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [46]:
evaluate(ground_truth, text_search)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.8466019417475729, 'mrr': 0.7127184466019416}

In [47]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [48]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.8679611650485437, 'mrr': 0.7363430420711974}

In [49]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [50]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/515 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8796116504854369, 'mrr': 0.7703559870550158}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8854368932038835, 'mrr': 0.7641747572815529}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8466019417475729, 'mrr': 0.7127184466019416}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8194174757281554, 'mrr': 0.6841100323624594}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7902912621359224, 'mrr': 0.6587055016181227}


In [51]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [52]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/515 [00:00<?, ?it/s]

In [53]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.961165,0.855631
4,1.0,2.0,0.2,0.959223,0.854207
3,1.0,2.0,0.1,0.959223,0.853236
19,2.0,4.0,0.2,0.959223,0.853236
35,5.0,10.0,0.5,0.959223,0.853236
20,2.0,4.0,0.5,0.959223,0.852654
6,1.0,4.0,0.1,0.959223,0.852265
18,2.0,4.0,0.1,0.957282,0.852201
34,5.0,10.0,0.2,0.957282,0.852039
33,5.0,10.0,0.1,0.957282,0.849612
